In [1]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_dataset("ailsntua/QEvasion")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")


# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )


# Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

# Focal Loss implementation
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

# ============ KEY CHANGES FOR LONGformer ============
# Change model checkpoint to Longformer
model_checkpoint = "allenai/longformer-base-4096"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # First tokenize normally
    tokenized = tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

    # ============ KEY CHANGE: Add global attention mask for Longformer ============
    # Longformer requires global attention on specific tokens for sequence classification
    # We set global attention to 1 only on the [CLS] token (first token)
    global_attention_mask = []
    for input_ids in tokenized['input_ids']:
        # Create mask: 1 for [CLS] token, 0 for all others
        mask = [1] + [0] * (len(input_ids) - 1)
        global_attention_mask.append(mask)

    tokenized['global_attention_mask'] = global_attention_mask
    return tokenized

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load Longformer model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# ============ OPTIONAL: Adjust batch size if memory issues ============
# Longformer might use more memory, so you might need to reduce batch size
# per_device_train_batch_size=4 instead of 8 if you encounter OOM errors

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# Training arguments - consider reducing batch size if you get memory errors
training_args = TrainingArguments(
    output_dir="Longformer_QEvasion_model",  # Changed output directory
    learning_rate=5e-5,
    per_device_train_batch_size=8,  # You might need to reduce this to 4 if you get CUDA out of memory
    per_device_eval_batch_size=8,   # Same for eval batch size
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,  # This helps with memory for long sequences
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    focal_gamma=1.0,
)

print(f"Using class weights: {class_weights}")
print("Starting Longformer training with Focal Loss...")
trainer.train()

print("Training completed!")

# Final evaluation
test_results = trainer.evaluate()
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})")
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Using class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Starting Longformer training with Focal Loss...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
1,0.768500,0.583544,0.301948,0.318340,0.233210,0.422343,0.486485,0.438042,0.301948
2,0.657600,0.599402,0.418831,0.419143,0.428166,0.435848,0.568156,0.524118,0.418831
3,0.527700,0.606582,0.409091,0.420687,0.406571,0.446280,0.568119,0.534871,0.409091
4,0.372900,0.610160,0.587662,0.538080,0.603750,0.525069,0.655222,0.587713,0.587662
5,0.263600,0.679535,0.587662,0.532417,0.601200,0.512032,0.631134,0.577177,0.587662


Training completed!



FINAL TEST RESULTS (from best epoch: Longformer_QEvasion_model/checkpoint-1724)
eval_loss: 0.6102
eval_accuracy: 0.5877
eval_f1_macro: 0.5381
eval_f1_weighted: 0.6038
eval_precision_macro: 0.5251
eval_precision_weighted: 0.6552
eval_recall_macro: 0.5877
eval_recall_weighted: 0.5877

Detailed predictions analysis (from best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.40      0.62      0.49        79
     Ambivalent       0.78      0.58      0.66       206
Clear Non-Reply       0.39      0.57      0.46        23

       accuracy                           0.59       308
      macro avg       0.53      0.59      0.54       308
   weighted avg       0.66      0.59      0.60       308


Confusion Matrix:
[[ 49  26   4]
 [ 71 119  16]
 [  3   7  13]]


In [2]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 8
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))



STARTING PART 2: Resuming training for 5 additional epochs...

Updated total number of epochs to: 8


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
6,0.281400,0.684170,0.548701,0.520409,0.565637,0.501874,0.639763,0.622264,0.548701
7,0.220400,0.706547,0.616883,0.566976,0.627668,0.537885,0.654949,0.632965,0.616883
8,0.143900,0.789907,0.639610,0.581211,0.648505,0.556949,0.666763,0.623745,0.639610


Additional 5 epochs of training completed!

FINAL TEST RESULTS (after 8 total epochs)
(Best model from all runs: Longformer_QEvasion_model/checkpoint-3448)


eval_loss: 0.7899
eval_accuracy: 0.6396
eval_f1_macro: 0.5812
eval_f1_weighted: 0.6485
eval_precision_macro: 0.5569
eval_precision_weighted: 0.6668
eval_recall_macro: 0.6237
eval_recall_weighted: 0.6396

Detailed predictions analysis (from new best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.46      0.54      0.50        79
     Ambivalent       0.77      0.67      0.72       206
Clear Non-Reply       0.44      0.65      0.53        23

       accuracy                           0.64       308
      macro avg       0.56      0.62      0.58       308
   weighted avg       0.67      0.64      0.65       308


Confusion Matrix:
[[ 43  33   3]
 [ 51 139  16]
 [  0   8  15]]
